# 03 - Feature engineering details

Ce notebook reprend la logique du fichier `feature_engineering.py`, mais de facon plus lisible pour comprendre les variables creees.


In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(BASE_DIR / 'data' / 'customer_churn.csv')
df.head()

,customer_id,gender,age,country,city,customer_segment,tenure_months,signup_channel,contract_type,monthly_logins,...,avg_resolution_time,complaint_type,csat_score,escalations,email_open_rate,marketing_click_rate,nps_score,survey_response,referral_count,churn
0,CUST_00001,Male,68,Bangladesh,London,SME,22,Web,Monthly,26,...,13.354360,Service,4.0,0,0.71,0.40,27,Satisfied,1,0
1,CUST_00002,Female,57,Canada,Sydney,Individual,9,Mobile,Monthly,7,...,25.140088,Billing,2.0,0,0.78,0.33,-19,Neutral,2,1
2,CUST_00003,Male,24,Germany,New York,SME,58,Web,Yearly,19,...,27.572928,Service,3.0,0,0.35,0.49,80,Neutral,1,0
3,CUST_00004,Male,49,Australia,Dhaka,Individual,19,Mobile,Yearly,34,...,26.420822,Technical,5.0,1,0.83,0.15,100,Neutral,0,0
4,CUST_00005,Male,65,Bangladesh,Delhi,Individual,52,Web,Monthly,20,...,26.674579,Technical,4.0,0,0.65,0.44,21,Unsatisfied,1,0


## Traitement des plaintes

Les valeurs manquantes de `complaint_type` signifient simplement qu il n y a pas eu de plainte connue. On les remplace donc par `Aucune Plainte`.

In [2]:
data = df.copy()
data['complaint_type'] = data['complaint_type'].fillna('Aucune Plainte')
data['has_complaint'] = (data['complaint_type'] != 'Aucune Plainte').astype(int)
data[['complaint_type', 'has_complaint']].head()

#list all count of complaint types
complaint_counts = data['complaint_type'].value_counts()
print(complaint_counts)

complaint_type
Technical         3498
Billing           2427
Aucune Plainte    2045
Service           2030
Name: count, dtype: int64


## Variables metier simples

On cree des indicateurs binaires faciles a expliquer : risque paiement, faible satisfaction, inactivite, contrat mensuel.

In [3]:
data['payment_risk'] = ((data['payment_failures'] > 0) | (data['price_increase_last_3m'] == 'Yes')).astype(int)
data['low_satisfaction'] = ((data['nps_score'] < 0) | (data['csat_score'] <= 2) | (data['survey_response'] == 'Unsatisfied')).astype(int)
data['inactive_customer'] = ((data['monthly_logins'] <= 5) | (data['last_login_days_ago'] >= 20)).astype(int)
data['monthly_contract'] = (data['contract_type'] == 'Monthly').astype(int)
data[['payment_risk', 'low_satisfaction', 'inactive_customer', 'monthly_contract']].head()

,payment_risk,low_satisfaction,inactive_customer,monthly_contract
0,1,0,0,1
1,1,1,0,1
2,1,0,1,0
3,0,0,1,0
4,0,1,0,1


## Ratios et scores

Les ratios permettent de comparer les clients plus justement. Par exemple, un client ancien avec 3 tickets support n est pas dans la meme situation qu un nouveau client avec 3 tickets.

In [4]:
tenure_safe = data['tenure_months'].clip(lower=1)
logins_safe = data['monthly_logins'].clip(lower=1)

data['tickets_per_tenure'] = data['support_tickets'] / tenure_safe
data['revenue_per_month'] = data['total_revenue'] / tenure_safe
data['fee_per_login'] = data['monthly_fee'] / logins_safe
data['support_pressure'] = data['support_tickets'] * data['avg_resolution_time']
data['engagement_score'] = (
    data['monthly_logins'] * 0.30
    + data['weekly_active_days'] * 2.0
    + data['features_used'] * 1.5
    + data['usage_growth_rate'] * 10
    - data['last_login_days_ago'] * 0.40
)
data['satisfaction_score'] = (
    data['csat_score'] * 10
    + data['nps_score'] * 0.20
    - data['escalations'] * 5
    - data['has_complaint'] * 8
)
data[['tickets_per_tenure', 'fee_per_login', 'engagement_score', 'satisfaction_score']].head()

,tickets_per_tenure,fee_per_login,engagement_score,satisfaction_score
0,0.181818,1.153846,27.1,37.4
1,0.111111,4.285714,10.0,8.2
2,0.017241,1.052632,16.8,38.0
3,0.157895,0.882353,15.9,57.0
4,0.000000,2.500000,18.6,36.2


## Verification rapide

On compare les moyennes selon `churn`. Le but est de voir si les variables creees apportent une separation entre clients qui restent et clients qui partent.

In [5]:
new_cols = ['has_complaint','payment_risk','low_satisfaction','inactive_customer','monthly_contract','tickets_per_tenure','fee_per_login','support_pressure','engagement_score','satisfaction_score']
data.groupby('churn')[new_cols].mean().round(3)

,has_complaint,payment_risk,low_satisfaction,inactive_customer,monthly_contract,tickets_per_tenure,fee_per_login,support_pressure,engagement_score,satisfaction_score
churn,,,,,,,,,,
0,0.796,0.502,0.517,0.192,0.496,0.086,3.365,28.987,16.891,31.375
1,0.793,0.587,0.667,0.344,0.502,0.185,6.381,28.894,15.402,26.281


## Quelles variables ont vraiment un signal ?

`payment_risk` regroupe deux conditions (`payment_failures > 0` et `price_increase_last_3m == 'Yes'`), mais elles n'ont pas le meme pouvoir predictif. On le verifie ci-dessous en comparant le taux de churn reel par sous-groupe.

In [6]:
print('--- Taux de churn par nombre d\'echecs de paiement ---')
display(df.groupby('payment_failures')['churn'].agg(['mean', 'count']))

print('\n--- Taux de churn : hausse de prix recente ---')
display(df.groupby('price_increase_last_3m')['churn'].agg(['mean', 'count']))

print('\n--- Taux de churn : plainte ou non ---')
display(data.groupby('has_complaint')['churn'].agg(['mean', 'count']))

print('\n--- Taux de churn : ancien (tenure) x satisfaction (csat) ---')
new_customer = (df['tenure_months'] <= 10).astype(int)
low_csat = (df['csat_score'] <= 2).astype(int)
display(df.groupby([new_customer.rename('nouveau_client'), low_csat.rename('csat_faible')])['churn'].agg(['mean', 'count']))

--- Taux de churn par nombre d'echecs de paiement ---


,mean,count
payment_failures,,
0,0.087607,6084
1,0.086651,2989
2,0.252564,780
3,0.215385,130
4,0.214286,14
5,0.333333,3



--- Taux de churn : hausse de prix recente ---


,mean,count
price_increase_last_3m,,
No,0.101676,8055
Yes,0.103856,1945



--- Taux de churn : plainte ou non ---


,mean,count
has_complaint,,
0,0.103178,2045
1,0.101823,7955



--- Taux de churn : ancien (tenure) x satisfaction (csat) ---


mean  count
nouveau_client csat_faible                 
0              0            0.051712   6981
               1            0.241935   1302
1              0            0.178862   1476
               1            0.336100    241

## Constat et pistes d'amelioration testees (non retenues)

Les chiffres ci-dessus confirment un point important pour la lecture des variables :

- **Signal reel et non lineaire** : `payment_failures` ne fait presque rien varier entre 0 et 1 echec (~8,6-8,7% de churn), mais **saute a 21-33% des 2 echecs**. `tenure_months` et `csat_score` ont aussi un vrai effet, et leur **combinaison** (client recent ET insatisfait) fait grimper le churn a **33,6%**, contre 5,2% pour un client ancien et satisfait.
- **Quasiment aucun signal** : `price_increase_last_3m` (10,2% vs 10,4% de churn) et `has_complaint` (10,3% vs 10,2%) ne separent presque pas les deux populations. Ce n'est pas une erreur de code : c'est une caracteristique de ce dataset (probablement synthetique), ou le churn n'a pas ete genere en fonction du prix ou des plaintes.

**Deux ameliorations ont ete testees** (en cross-validation repetee, 20 entrainements XGBoost par variante, seuil metier 0.20, memes hyperparametres que le notebook `04_entrainement_complet.ipynb`) :

1. Recalibrer `payment_risk` sur `payment_failures >= 2` (au lieu de `> 0`) : **aucun gain** (recall 86,16% vs 86,12%, dans l'ecart-type de ±2,2 points), et legerement plus de faux positifs.
2. Ajouter une variable d'interaction explicite `nouveau_client x faible_satisfaction` : **aucun gain net** (recall +0,46 point, toujours dans le bruit, avec plus de faux positifs en contrepartie).

**Pourquoi ces pistes n'apportent rien** : le modele retenu est un XGBoost (arbres de decision boostes). Un arbre peut deja apprendre seul un seuil (`payment_failures >= 2`) ou une interaction (`tenure_months` puis `csat_score` sur la meme branche) directement a partir des colonnes brutes, qui restent presentes dans le jeu de variables. Prefabriquer ces combinaisons en une seule colonne n'apporte donc pas d'information nouvelle a ce type de modele — cela aiderait davantage un modele lineaire (Logistic Regression), qui ne peut pas decouvrir ces interactions tout seul.

**Conclusion sur ces deux pistes** : elles n'ont pas ete retenues (aucune modification du code sur `payment_risk` ou une interaction). Une autre piste, la redondance entre variables, a en revanche donne lieu a une correction du code : voir la section suivante.</cell id="e7ffcb8a">


## Redondance entre variables creees (corrigee)

Une revue de correlation entre les variables metier creees (au-dela de la simple lecture des noms) revele de vraies redondances.

In [7]:
redundancy_cols = [
    'has_complaint', 'payment_risk', 'low_satisfaction', 'inactive_customer',
    'monthly_contract', 'tickets_per_tenure', 'revenue_per_month', 'fee_per_login',
    'support_pressure', 'engagement_score', 'satisfaction_score',
]
corr = data[redundancy_cols].corr()

print("--- revenue_per_month vs monthly_fee (colonne brute, pas dans 'data' d'origine) ---")
print(pd.concat([data['revenue_per_month'], df['monthly_fee']], axis=1).corr())

print('\n--- Paires les plus correlees parmi les variables creees ---')
pairs = corr.abs().unstack().sort_values(ascending=False)
pairs = pairs[pairs < 1.0]
seen = set()
for (a, b), v in pairs.items():
    key = tuple(sorted([a, b]))
    if key in seen:
        continue
    seen.add(key)
    print(f'{a:20s} vs {b:20s} : {corr.loc[a, b]:+.3f}')
    if len(seen) >= 5:
        break

--- revenue_per_month vs monthly_fee (colonne brute, pas dans 'data' d'origine) ---
                   revenue_per_month  monthly_fee
revenue_per_month                1.0          1.0
monthly_fee                      1.0          1.0

--- Paires les plus correlees parmi les variables creees ---
low_satisfaction     vs satisfaction_score   : -0.493
engagement_score     vs inactive_customer    : -0.434
inactive_customer    vs fee_per_login        : +0.373
revenue_per_month    vs fee_per_login        : +0.308
support_pressure     vs tickets_per_tenure   : +0.306


**Constat** :

- `revenue_per_month` est un **doublon exact de `monthly_fee`** (correlation = 1.000), car `total_revenue = monthly_fee * tenure_months` dans ce dataset : `revenue_per_month = total_revenue / tenure_months = monthly_fee`. C'est un vrai defaut de conception, pas une redondance acceptable.
- `low_satisfaction` (flag binaire) et `satisfaction_score` (score continu) portent largement le meme signal (corr -0,49) : memes composantes (CSAT, NPS, plainte).
- `inactive_customer` (flag binaire) et `engagement_score` (score continu) portent egalement le meme signal (corr -0,43) : memes composantes (logins, recence de connexion).

**Correction appliquee** dans `feature_engineering.py` : suppression de `revenue_per_month`, `low_satisfaction` et `inactive_customer`. On conserve les versions continues (`satisfaction_score`, `engagement_score`), plus nuancees que les flags binaires qu'elles remplacent.

**Verification avant application** (cross-validation repetee, 20 entrainements XGBoost, seuil metier 0.20) : le recall reste stable (0,861 avant nettoyage vs 0,862 apres, dans le bruit) — ce nettoyage simplifie le code et les graphiques d'interpretabilite (SHAP, feature importance) sans perte de performance.

**Consequence sur le choix du modele final** : ce nettoyage a fait apparaitre une egalite parfaite de recall entre XGBoost et Random Forest (178/204 vrais positifs chacun) sur le split de test du notebook `04_entrainement_complet.ipynb`. La regle de selection automatique a donc ete renforcee : en cas d'egalite de recall (tolerance 1 point), le modele le plus rapide/sobre a entrainer est desormais prefere plutot que de departager uniquement sur la precision brute — ce qui confirme XGBoost (1,71 s contre 10,05 s pour Random Forest, cf. notebook 04 section 9).